In [ ]:
# Cell 1: ORS Setup and Geocoder Pipeline
import sys
!{sys.executable} -m pip install -q pandas numpy openrouteservice requests

import pandas as pd
import numpy as np
import openrouteservice
from openrouteservice import client

# Replace with your actual OpenRouteService API key
ORS_API_KEY = "YOUR_ORS_API_KEY_HERE"
ors_client = client.Client(key=ORS_API_KEY)

# 1. Load the Bengaluru Dataset
df = pd.read_csv("data/Banglore_traffic_Dataset.csv")

# Ensure we have clean strings for geocoding
df['Search_Query'] = df['Road/Intersection Name'] + ", " + df['Area Name'] + ", Bengaluru, India"

def geocode_intersection(query):
    """Uses ORS Pelias Geocoding API to resolve text to Lat/Lon"""
    try:
        res = ors_client.pelias_search(text=query)
        # Extract the first matching coordinate [Longitude, Latitude]
        coords = res['features'][0]['geometry']['coordinates']
        return coords[0], coords[1]
    except Exception as e:
        return None, None

print("Geocoding first 10 nodes for the MVP...")
# We limit to 10 nodes for API safety; scale up for the full dataset later
df_mvp = df.head(10).copy()
df_mvp['Longitude'], df_mvp['Latitude'] = zip(*df_mvp['Search_Query'].apply(geocode_intersection))

# Drop any failed geocodes
df_mvp = df_mvp.dropna(subset=['Longitude', 'Latitude']).reset_index(drop=True)
print(f"Successfully geocoded {len(df_mvp)} intersections.")
df_mvp[['Area Name', 'Road/Intersection Name', 'Latitude', 'Longitude']]

[Block 2] ORS Matrix Calculation & Just-In-Time Synthesizer

In [ ]:
# Cell 2: True ORS Travel Time Matrix & Diurnal Curve Synthesizer

# 1. Fetch real-world baseline Distance & Duration matrices from ORS
locations = df_mvp[['Longitude', 'Latitude']].values.tolist()

print("Fetching ORS Distance Matrix...")
matrix_response = ors_client.distance_matrix(
    locations=locations,
    profile='driving-car',
    metrics=['distance', 'duration']
)

base_distances = np.array(matrix_response['distances']) # in meters
base_durations = np.array(matrix_response['durations']) # in seconds

# 2. The Just-In-Time Gaussian Traffic Synthesizer
def get_dynamic_travel_time(base_duration_matrix, congestion_series, incidents_series, time_of_day_hours):
    """
    Applies the Bimodal Gaussian traffic penalty directly to the ORS time matrix.
    time_of_day_hours: float between 0.0 and 24.0 (e.g., 14.5 is 2:30 PM)
    """
    mu_1, mu_2, sigma = 9.5, 18.5, 1.5
    
    # Calculate Gaussian rush-hour intensity (0.0 to 1.0)
    rush_factor = np.exp(-((time_of_day_hours - mu_1)**2) / (2 * sigma**2)) + \
                  np.exp(-((time_of_day_hours - mu_2)**2) / (2 * sigma**2))
    
    dynamic_durations = np.copy(base_duration_matrix)
    
    for i in range(len(congestion_series)):
        # Normalize congestion from 1-100 to a 0.0 - 0.5 multiplier
        c_factor = (congestion_series.iloc[i] / 100.0) * 0.5
        
        # Immediate Incident Impact (Jarvis Trigger)
        incident_penalty = 1.5 if incidents_series.iloc[i] > 0 else 1.0
        
        # Calculate localized temporal penalty
        temporal_penalty = 1.0 + (c_factor * rush_factor * incident_penalty)
        
        # Apply to all edges arriving AT node i
        dynamic_durations[:, i] *= temporal_penalty
        
    return dynamic_durations

# Test the Synthesizer
t_11am = get_dynamic_travel_time(base_durations, df_mvp['Congestion Level'], df_mvp['Incident Reports'], 11.0)
t_6pm = get_dynamic_travel_time(base_durations, df_mvp['Congestion Level'], df_mvp['Incident Reports'], 18.5)

print("Average edge travel time at 11:00 AM:", round(np.mean(t_11am), 2), "seconds")
print("Average edge travel time at  6:30 PM:", round(np.mean(t_6pm), 2), "seconds")

[Block 3] The Jarvis-Inspired Adaptive Swarm
This engine listens for environmental shifts. When a route changes mid-execution, we use "Vehicular Inspired Quantum Swarms" to preserve the fleet's trajectory and only dynamically re-solve the remaining nodes.

In [ ]:
# Cell 3: Jarvis Dynamic QPSO Mechanism

class JarvisAdaptiveEngine:
    def __init__(self, current_time, base_durations, df_nodes):
        self.current_time = current_time
        self.base_durations = base_durations
        self.df_nodes = df_nodes
        self.n_nodes = len(df_nodes)
        
        # Initial Quantum State
        self.theta = np.random.uniform(0, np.pi/2, (30, self.n_nodes))
        self.update_environment()

    def update_environment(self, time_shift=0.0, new_incidents=None):
        """Jarvis trigger: Updates the environmental tensors and resets local bests."""
        self.current_time += time_shift
        
        if new_incidents is not None:
            # Overwrite incident reports dynamically
            for idx, count in new_incidents.items():
                self.df_nodes.at[idx, 'Incident Reports'] = count
                print(f"J.A.R.V.I.S Alert: New incident reported at {self.df_nodes.at[idx, 'Road/Intersection Name']}!")
                
        # Re-calculate the dynamic matrix based on new time/incidents
        self.dynamic_matrix = get_dynamic_travel_time(
            self.base_durations, 
            self.df_nodes['Congestion Level'], 
            self.df_nodes['Incident Reports'], 
            self.current_time
        )
        
        # In the Jarvis framework, we DO NOT reset the swarm completely.
        # We retain the global best (the elite vehicular prior) but reset personal bests
        # to force the quantum wave-function to explore local detours.
        self.pbest_cost = np.full(30, np.inf)

    def evaluate_cost(self, keys):
        # A simple Traveling Salesman formulation for the MVP adaptive test
        order = np.argsort(keys)
        cost = 0.0
        for i in range(len(order) - 1):
            cost += self.dynamic_matrix[order[i], order[i+1]]
        return cost

    def step_swarm(self):
        """Executes a single step of the Quantum Rotation Gate."""
        pos = np.sin(self.theta)**2
        
        for i in range(30):
            c = self.evaluate_cost(pos[i])
            if c < self.pbest_cost[i]:
                self.pbest_cost[i] = c
                
        gbest_idx = np.argmin(self.pbest_cost)
        gbest_cost = self.pbest_cost[gbest_idx]
        return gbest_cost, pos[gbest_idx]

# --- Simulation Run ---
jarvis = JarvisAdaptiveEngine(current_time=16.0, base_durations=base_durations, df_nodes=df_mvp) # Start at 4:00 PM
print("\n--- Initiating Baseline Route (4:00 PM) ---")
best_cost_4pm, _ = jarvis.step_swarm()
print(f"Optimized Travel Time: {round(best_cost_4pm/60, 2)} minutes")

# Fast forward to 6:30 PM (Rush hour) AND a sudden crash at node 2 (Marathahalli Bridge)
print("\n--- Jarvis Environment Shift Detected ---")
jarvis.update_environment(time_shift=2.5, new_incidents={2: 5}) # 5 incidents reported
best_cost_630pm, _ = jarvis.step_swarm()

print(f"Re-Optimized Travel Time with Rush Hour + Crash: {round(best_cost_630pm/60, 2)} minutes")